# 21 · Partitioning benchmark and small-file compaction (Part D, step D4)

**Why:** a spatial join (every populated cell × every school within a distance) is the core of D5. How Spark splits
the data over its tasks decides whether that join takes seconds or hours; this notebook measures it on real data.

**1. Small files.** The D2 table was written by 120 map tasks, one file per task and state → 6,210 small files.
Many small files slow every later read (each file has a fixed opening cost). The table is rewritten with one
shuffle (all cells of a state together, ≤ 5 M rows per file); read time is measured before and after; cell
counts and population totals must be identical before the compacted copy replaces the original.

**2. Join strategies** — count every (cell, school) pair within **1 km** (the primary-school walking norm,
`ST_DWithin(…, 1000, true)` = distance on the WGS84 ellipsoid in metres), in **Kerala** (2.98 M cells) and **Uttar
Pradesh** (13.75 M cells, 149 k schools):

| Strategy | How the work is split |
|---|---|
| Sedona join, **KDB-tree** | both sides cut into spatial partitions by a KDB-tree built from a sample (balanced counts) |
| Sedona join, **Quad-tree** | same, with a quad-tree grid |
| **Broadcast index join** | the smaller side (schools) is copied to every task with an R-tree index; cells are not moved |
| **H3 hexagon equi-join** | cells and schools get Uber H3 hexagon ids (res 7); an ordinary hash join on the id + exact distance check |

Each strategy runs 3 times in Kerala and once in Uttar Pradesh (median reported), on cached inputs; a strategy still
running after 10 minutes is stopped and recorded as too slow; with the **same pair count** required from all
four; the join operator Spark chose, the number of output partitions and their skew (largest ÷ mean) are recorded.

**Outputs:** `data/lake/gold/partition_benchmark.csv` (summary), `partition_benchmark_runs.csv` (every run); the
compacted `silver/population_cells`.

## Project folder (laptop or Colab)

In [1]:
# Project folder. Laptop: open the notebook from "PROJECTS IPYNB" (or the project root).
# Google Colab: upload the project folder to Google Drive ("My Drive") and set its name below.
from pathlib import Path
import sys
PROJECT_FOLDER = "Distance-Capacity-and-Inequality-in-Access-to-Government-and-Aided-Schools"  # Colab only: folder name inside My Drive
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_FOLDER
else:
    PROJECT = Path.cwd().parent if Path.cwd().name in ("notebooks", "PROJECTS IPYNB") else Path.cwd()
assert (PROJECT / "scripts").is_dir(), f"project folder not found: {PROJECT}"
sys.path.insert(0, str(PROJECT / "scripts"))

In [2]:
# Colab only: install packages that Colab does not have by default (does nothing on the laptop)
if IN_COLAB:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "rapidfuzz", "pymupdf", "pyogrio", "rasterio"], check=True)

## Install the Part D packages

In [3]:
# Part D packages, pinned in requirements-bigdata.txt (PySpark 3.5.9, apache-sedona 1.9.1, pyarrow).
# Colab also needs Java 17, which Spark runs on (the laptop already has it: JAVA_HOME).
import os
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT / "requirements-bigdata.txt")], check=True)
if IN_COLAB:
    subprocess.run("apt-get install -y -qq openjdk-17-jdk-headless > /dev/null", shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

## Start Spark + Sedona

In [4]:
import shutil
import statistics
import time

import pandas as pd
from pyspark.sql import functions as F
from bigdata.spark_session import get_spark

sys.path.insert(0, str(PROJECT / "scripts" / "bigdata"))
import population_cells as pc

SILVER = PROJECT / "data" / "lake" / "silver"
GOLD = PROJECT / "data" / "lake" / "gold"
GOLD.mkdir(parents=True, exist_ok=True)
CELLS = SILVER / "population_cells"
COMPACT = SILVER / "population_cells_compact"
free_gb = shutil.disk_usage(PROJECT).free / 1e9
print(f"free disk: {free_gb:.1f} GB")
assert free_gb > 7, "compaction needs ~2 GB for the copy + ~2 GB shuffle: stop"
spark = get_spark(PROJECT, cores=6, memory="6g")  # 6 cores: the laptop's free RAM (see notebook 17)
LON, LAT = pc.lonlat_sql(PROJECT / "data" / "raw" / "worldpop")
print("cores:", spark.sparkContext.defaultParallelism, "| Spark UI:", spark.sparkContext.uiWebUrl)
results = []


def timed(label, workload, fn, runs=3):
    """Run fn() (returns a number) `runs` times; record every run; return the median seconds and the value."""
    secs, val = [], None
    for r in range(runs):
        t = time.time()
        val = fn()
        secs.append(time.time() - t)
        results.append({"workload": workload, "strategy": label, "run": r + 1, "seconds": round(secs[-1], 2), "result": val})
        if r == 0 and secs[0] > 300:  # a very slow first run: do not repeat it
            break
    print(f"{workload:22s} {label:34s} median {statistics.median(secs):7.1f} s  ({len(secs)} runs)  result {val:,}")
    return statistics.median(secs), val

free disk: 18.2 GB


cores: 6 | Spark UI: http://127.0.0.1:4040


## 1. Small files: compact the population-cells table

In [5]:
# 1. Small files: the D2 table was written straight from 120 map tasks -> 6,210 small files.
import json
DONE = GOLD / "compaction_benchmark.json"
if DONE.exists() and len(list(CELLS.rglob("*.parquet"))) < 100:  # compacted on an earlier run: reuse its measurements
    done = json.loads(DONE.read_text())
    print("table already compacted on an earlier run:", done["files_before"], "->", done["files_after"], "files in",
          done["compaction_minutes"], "min")
    for r in done["runs"]:
        print(f"  {r['strategy']:30s} median {r['median_seconds']} s ({r['runs']} runs)")
        results.append({"workload": r["workload"], "strategy": r["strategy"], "run": "median", "seconds": r["median_seconds"],
                        "result": r["result"]})
    raise_skip = True
else:
    raise_skip = False
if not raise_skip:
  files_before = len(list(CELLS.rglob("*.parquet")))
  read_sum = lambda path: lambda: int(spark.read.parquet(str(path)).agg(F.count("*")).first()[0])
  timed(f"read + count, {files_before:,} files", "population_cells", read_sum(CELLS))

  t = time.time()
  (spark.read.parquet(str(CELLS)).repartition("state")      # one shuffle: all cells of a state to one task
        .write.mode("overwrite").option("compression", "zstd").option("maxRecordsPerFile", 5_000_000)
        .partitionBy("state").parquet(str(COMPACT)))
  files_after = len(list(COMPACT.rglob("*.parquet")))
  print(f"compacted in {(time.time() - t) / 60:.1f} min: {files_before:,} -> {files_after:,} files")
  timed(f"read + count, {files_after:,} files", "population_cells", read_sum(COMPACT))

  agg = lambda p: spark.read.parquet(str(p)).agg(F.count("*").alias("cells"), F.countDistinct("cell_id").alias("ids"),
                                                 *[F.sum(c).alias(c) for c in pc.BANDS]).first().asDict()
  a, b = agg(CELLS), agg(COMPACT)
  assert a["cells"] == b["cells"] == b["ids"], (a, b)
  assert all(abs(a[c] - b[c]) <= 1e-9 * a[c] for c in pc.BANDS), (a, b)
  print("same cells and totals:", {k: (round(v / 1e6, 2) if isinstance(v, float) else v) for k, v in b.items()})

  # swap: the compacted copy becomes the silver table
  old = SILVER / "population_cells_old"
  shutil.rmtree(old, ignore_errors=True)
  CELLS.rename(old)
  COMPACT.rename(CELLS)
  shutil.rmtree(old)
  print("silver/population_cells is now the compacted table;", len(list(CELLS.rglob("*.parquet"))), "files")
  DONE.write_text(json.dumps({"files_before": files_before, "files_after": files_after,
                              "compaction_minutes": None, "runs": [{**r, "median_seconds": r["seconds"], "runs": 1}
                                                                   for r in results if r["workload"] == "population_cells"]}))

table already compacted on an earlier run: 6210 -> 53 files in 8.9 min
  read + count, 6,210 files      median 6.0 s (3 runs)
  read + count, 53 files         median 0.6 s (3 runs)


## 2. Join workloads

In [6]:
# 2. Join workloads: cells x schools within 1 km (the primary-school walking norm), Kerala and Uttar Pradesh.
#    Inputs are cached first, so the timings measure the join only.
def workload(state):
    cells = spark.read.parquet(str(CELLS)).filter(F.col("state") == state) \
                 .selectExpr("cell_id", f"ST_Point({LON}, {LAT}) AS geom").cache()
    schools = spark.read.format("geoparquet").load(str(SILVER / "schools_india")) \
                   .filter((F.col("state_name") == state) & F.col("geometry").isNotNull()) \
                   .selectExpr("udise_code", "geometry AS geom").cache()
    print(f"{state}: {cells.count():,} cells x {schools.count():,} schools")
    return cells, schools


STATES = ["Kerala", "Uttar Pradesh"]

### The four strategies

In [7]:
def plan_ops(df):
    """Join operator Spark/Sedona chose, read from the physical plan."""
    plan = df._jdf.queryExecution().executedPlan().toString()
    ops = ["BroadcastIndexJoin", "DistanceJoin", "RangeJoin", "BroadcastHashJoin", "SortMergeJoin", "ShuffledHashJoin",
           "BroadcastNestedLoopJoin", "CartesianProduct"]
    return ", ".join(o for o in ops if o in plan)


def sedona_grid(cells, schools, grid):
    spark.conf.set("sedona.join.gridtype", grid)                 # how Sedona cuts space into partitions
    spark.conf.set("sedona.join.autoBroadcastJoinThreshold", "-1")  # force a partitioned (not broadcast) join
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    cells.createOrReplaceTempView("c")
    schools.createOrReplaceTempView("s")
    return spark.sql("SELECT c.cell_id, s.udise_code FROM c JOIN s ON ST_DWithin(c.geom, s.geom, 1000, true)")


def broadcast(cells, schools):
    cells.createOrReplaceTempView("c")
    schools.createOrReplaceTempView("s")
    return spark.sql("SELECT /*+ BROADCAST(s) */ c.cell_id, s.udise_code FROM c JOIN s ON ST_DWithin(c.geom, s.geom, 1000, true)")


def h3(cells, schools):
    # H3 res 7 hexagons (edge ~1.2 km): any cell within 1 km of a school lies in the school's hexagon or a neighbour,
    # so an ordinary equi-join on the hexagon id + an exact distance check finds every pair.
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
    ch = cells.selectExpr("cell_id", "geom AS cgeom", "ST_H3CellIDs(geom, 7, false)[0] AS h3")
    sh = schools.selectExpr("udise_code", "geom AS sgeom", "explode(ST_H3KRing(ST_H3CellIDs(geom, 7, false)[0], 1, false)) AS h3")
    return ch.join(sh, "h3").where("ST_DistanceSpheroid(cgeom, sgeom) <= 1000").select("cell_id", "udise_code")


STRATEGIES = {"Sedona join, KDB-tree partitions": lambda c, s: sedona_grid(c, s, "kdbtree"),
              "Sedona join, Quad-tree partitions": lambda c, s: sedona_grid(c, s, "quadtree"),
              "Broadcast index join (schools)": broadcast,
              "H3 hexagon equi-join": h3}

### Run and compare

In [8]:
import threading

LIMIT_S = 600          # a strategy still running after 10 minutes is stopped and recorded as too slow
RUNS = {"Kerala": 3, "Uttar Pradesh": 1}  # UP: one run per strategy (each run takes minutes)
parts = {}


def capped(df, key):
    """Count the join's pairs per output partition in one pass (total pairs + skew); cancel after LIMIT_S."""
    sc = spark.sparkContext
    sc.setJobGroup(key, key, interruptOnCancel=True)
    timer = threading.Timer(LIMIT_S, lambda: sc.cancelJobGroup(key))
    timer.start()
    try:
        parts[key] = df.groupBy(F.spark_partition_id().alias("p")).count().toPandas()["count"]
    finally:
        timer.cancel()
    return int(parts[key].sum())


# Uttar Pradesh: lightest strategies first. On this laptop a KDB-tree join of UP ran out of memory and stopped Spark
# (Java cannot be restarted in the same notebook), so a failure ends the UP runs and the rest are marked "not run".
ORDER = {"Kerala": list(STRATEGIES), "Uttar Pradesh": ["Broadcast index join (schools)", "H3 hexagon equi-join",
                                                       "Sedona join, Quad-tree partitions", "Sedona join, KDB-tree partitions"]}
summary, spark_ok = [], True
for state in STATES:
    cells, schools = workload(state)  # one state's inputs in memory at a time
    for name in ORDER[state]:
        if not spark_ok:
            summary.append({"workload": state, "strategy": name, "note": "not run: Spark stopped after the previous failure"})
            continue
        df = STRATEGIES[name](cells, schools)
        ops = plan_ops(df)
        assert ops and "NestedLoop" not in ops and "Cartesian" not in ops, f"{name}: no spatial join in the plan ({ops})"
        key = f"{state} | {name}"
        try:
            med, pairs = timed(name, state, lambda: capped(df, key), runs=RUNS[state])
            p = parts[key]
            row = {"median_seconds": round(med, 1), "pairs": pairs, "partitions": len(p),
                   "skew_max_over_mean": round(p.max() / p.mean(), 2), "note": ""}
        except Exception as e:  # cancelled after LIMIT_S, or Spark ran out of memory
            print(f"{state:22s} {name:34s} stopped after {LIMIT_S // 60} min ({type(e).__name__})")
            row = {"median_seconds": None, "pairs": None, "partitions": None, "skew_max_over_mean": None,
                   "note": f"too slow or out of memory: not finished in {LIMIT_S // 60} min"}
            spark_ok = False
        summary.append({"workload": state, "strategy": name, "join_operator": ops} | row)
    if spark_ok:
        spark.conf.unset("sedona.join.gridtype")
        cells.unpersist(); schools.unpersist()
summary = pd.DataFrame(summary)
print(summary.to_string(index=False))
for state, g in summary.dropna(subset=["pairs"]).groupby("workload"):
    assert g["pairs"].nunique() == 1, f"{state}: strategies disagree on the pair count {g['pairs'].tolist()}"
print("every strategy finds exactly the same pairs")

Kerala: 2,977,966 cells x 11,767 schools


Kerala                 Sedona join, KDB-tree partitions   median     8.9 s  (3 runs)  result 4,041,560


Kerala                 Sedona join, Quad-tree partitions  median     9.1 s  (3 runs)  result 4,041,560


Kerala                 Broadcast index join (schools)     median     7.2 s  (3 runs)  result 4,041,560


Kerala                 H3 hexagon equi-join               median    16.7 s  (3 runs)  result 4,041,560


Uttar Pradesh: 13,754,598 cells x 149,506 schools


Uttar Pradesh          Broadcast index join (schools)     median    25.1 s  (1 runs)  result 34,660,289


Uttar Pradesh          H3 hexagon equi-join               median   102.9 s  (1 runs)  result 34,660,289


Uttar Pradesh          Sedona join, Quad-tree partitions  median    71.5 s  (1 runs)  result 34,660,289


Uttar Pradesh          Sedona join, KDB-tree partitions   stopped after 10 min (Py4JJavaError)
     workload                          strategy      join_operator  median_seconds      pairs  partitions  skew_max_over_mean                                              note
       Kerala  Sedona join, KDB-tree partitions       DistanceJoin             8.9  4041560.0         9.0                1.78                                                  
       Kerala Sedona join, Quad-tree partitions       DistanceJoin             9.1  4041560.0        14.0                2.77                                                  
       Kerala    Broadcast index join (schools) BroadcastIndexJoin             7.2  4041560.0         1.0                1.00                                                  
       Kerala              H3 hexagon equi-join      SortMergeJoin            16.7  4041560.0         6.0                1.09                                                  
Uttar Pradesh    Broadcas

every strategy finds exactly the same pairs


## Save and conclude

In [9]:
pd.DataFrame(results).to_csv(GOLD / "partition_benchmark_runs.csv", index=False)
summary.to_csv(GOLD / "partition_benchmark.csv", index=False)
done = summary.dropna(subset=["median_seconds"])
best = done.loc[done.groupby("workload")["median_seconds"].idxmin(), ["workload", "strategy", "median_seconds"]]
print("fastest per workload:")
print(best.to_string(index=False))
if spark_ok:
    spark.stop()

fastest per workload:
     workload                       strategy  median_seconds
       Kerala Broadcast index join (schools)             7.2
Uttar Pradesh Broadcast index join (schools)            25.1
